In [ ]:
import os, copy
import glob

import numpy as np
import scipy.optimize as so
import pandas as pd
import xarray as xr
import rioxarray as rxr
import datetime as dt

import netCDF4
import h5py
from osgeo import gdal

%matplotlib inline  
import matplotlib as mpl
import matplotlib.pyplot as plt
import colorcet as cc

import panel as pn
pn.extension()
opj = os.path.join


In [ ]:
workdir = '/sat_data/satellite/acix-iii/results'
workdir='/media/harmel/TOSHIBA EXT/acix-iii'
#workdir='/media/harmel/TOSHIBA EXT/data/satellite/prisma/zoffoli/L2A'
#workdir='/DATA/git/satellite_app/hgrs/'
files = pn.widgets.FileSelector(workdir)

files
aeronet_site = 'South_Greenbay'
aeronet_site = 'Galata_Platform'

aeronet_site = 'Venise'

aeronet_site = 'Bahia_Blanca'
site='bahiablanca'
 

sites=[  ['ariaketower','ARIAKE_TOWER'],
         ['bahiablanca','Bahia_Blanca'],
         ['casablanca','Casablanca_Platform'],
         ['galataplatform','Galata_Platform'],
         ['gustavdalentower','Gustav_Dalen_Tower'],
         ['irbelighthouse','Irbe_Lighthouse'],
         ['kemigawa','Kemigawa'],
         ['lakeerie','Lake_Erie'],
         ['lakeokeechobee','Lake_Okeechobee'],
         ['lisco','LISCO'],
         ['lucinda','Lucinda'],
         ['palgrunden','Palgrunden'],
         ['sanmarcoplatform','San_Marco_Platform'],
         ['section7','Section-7_Platform'],
         ['socheongcho','Socheongcho'],
         ['southgreenbay','South_Greenbay'],
         ['uscseaprism','USC_SEAPRISM_2'],
         ['venezia','Venise'],
         ['wavecissite','WaveCIS_Site_CSI_6'],
         ['zeebrugge','Zeebrugge-MOW1']]
site,aeronet_site = sites[0]


In [ ]:

from aeronet_visu import data_loading as dl
opj = os.path.join
idir = '/DATA/AERONET/OCv3/'
figdir= '/DATA/AERONET/fig'


file = aeronet_site+'_OCv3.lev15'

irr = dl.irradiance()
irr.load_F0()

params = ['Lwn','Lwn_IOP','Lwn_f/Q']

# ---------------------------------------------
# Load data and convert into xarray
# ---------------------------------------------

df = dl.read(opj(idir, file)).read_aeronet_ocv3()

df = df.droplevel(0, 1)
# criteria to select scalar and spectrum values
criteria = df.columns.get_level_values(1) == ''
df_att = df.loc[:, criteria].droplevel(1, 1)
df_spec = df.loc[:, ~criteria]

ds = df_spec.stack().to_xarray()
ds = xr.merge([ds, df_att.to_xarray()])
del df

ds = ds.assign_coords({'level_1': ds.level_1.astype(float)}).rename({'level_1': "wl"})
ds['SZA'] = ds.Solar_Zenith_Angle.mean(axis=1)
ds['year']=ds['date.year']
ds['season']=ds['date.season']

wl = ds.wavelength * 1000
for param in params:
    ds['Rrs_'+param] = ds[param] / (irr.get_F0(wl) * 0.1)
ds=ds.sortby("wl")

In [ ]:
lat,lon = ds['Site_Latitude(Degrees)'].mean().values,ds['Site_Longitude(Degrees)'].mean().values
print(lat,lon)

In [ ]:

for file in glob.glob(opj(workdir,site,'*.nc')):
    print(file)
                      

In [ ]:
dc=[]
for file in glob.glob(opj(workdir,site,'*.nc')):
    print(file)
    img =xr.open_dataset(file)
    date = dt.datetime.strptime(img.acquisition_date,'%Y-%m-%dT%H:%M:%S.%f')
    #img.acquisition_date

    Ttot_Ed=xr.open_dataset('/DATA/git/satellite_app/hgrs/data/lut/transmittance_downward_irradiance.nc')
    sza=np.nanmean(img.sza)
    vza= np.nanmean(img.vza)
    aot_ref= np.nanmean(img.aot_ref)
    model = img.aerosol_model
    wl =img.wl

    Ttot_Ed_ = Ttot_Ed.Ttot_Ed.sel(model=model).interp(sza=sza, method='cubic').interp(aot_ref=aot_ref, method='quadratic').interp(wl=wl, method='cubic')
    Ttot_Lu_ = Ttot_Ed.Ttot_Ed.sel(model=model).interp(sza=vza, method='cubic').interp(aot_ref=aot_ref, method='quadratic').interp(wl=wl, method='cubic')**1.05
    Ttot = (Ttot_Ed_ *Ttot_Lu_).reset_coords(drop=True)
    param = 'Rrs' #Rtoa'
    #img = prod[['Rtoa','Ltoa']] 
    img['Rrs_corr'] = img[param]/Ttot

    dc.append(img)

## Plot and interact

In [ ]:
def find_nearest(arr,lon,lat):
    abslat = np.abs(arr.lat-lat)
    abslon= np.abs(arr.lon-lon)
    dist = np.maximum(abslon,abslat)
    return np.unravel_index(dist.argmin(),arr.lon.shape)
    #return np.unravel_index(np.abs(arr - val).argmin(),arr.shape)
find_nearest(dc[0],float(lon),float(lat))


In [ ]:
nrows=len(dc)//4 +1 
fig,axs = plt.subplots(nrows,4,figsize=(20,3*nrows),sharey=True,sharex=True)
axs=axs.ravel()
for ax in axs:
    ax.set_visible(False)
for i_, img in enumerate(dc):
    axs[i_].set_visible(True)
    axs[i_].minorticks_on()
    date= dt.datetime.strptime(img.acquisition_date,'%Y-%m-%dT%H:%M:%S.%f')
    
    xcenter,ycenter=find_nearest(img,lon,lat)
    for i in range(6):
        for j in range(6):
            img.Rrs.isel(x=xcenter+i,y=ycenter+j).plot(x='wl',color='grey',alpha=0.5,lw=0.7,ax=axs[i_])#,label='PRISMA')
            img.Rrs_corr.isel(x=xcenter+i,y=ycenter+j).plot(x='wl',color='black',alpha=0.5,lw=0.7,ax=axs[i_])#,label='PRISMA')
    for param in params:
        ds['Rrs_'+param].sel(date=str(date),method='nearest').dropna('wl').plot(x='wl',marker='o',ms=4,label=param,ax=axs[i_])
    axs[i_].set_ylabel('$R_{rs}\ (sr^{-1})$')
    axs[i_].hlines(0,380,1120,ls=':',lw=0.5,color='black',zorder=0)

    axs[i_].legend(fontsize=10)
#plt.xlim(390,1100)

In [ ]:
hours=6
delta = dt.timedelta(hours=hours/2)


In [ ]:
fig,axs = plt.subplots(nrows,4,figsize=(20,3*nrows),sharey=True,sharex=True)
axs=axs.ravel()
for ax in axs:
    ax.set_visible(False)
for i_, img in enumerate(dc):
    axs[i_].set_visible(True)
    axs[i_].minorticks_on()
    date= dt.datetime.strptime(img.acquisition_date,'%Y-%m-%dT%H:%M:%S.%f')
    p = ds['Aerosol_Optical_Depth'].sel(date=slice(date-delta,date+delta)).dropna('wl')
    if len(p.date) > 0:
        p.plot(x='wl',marker='o',ms=4,hue='date',lw=0.7,ax=axs[i_],add_legend=False)
   
    for i in range(6):
        for j in range(6):
            axs[i_].plot(550,img.aot_ref_full.isel(x=xcenter+i,y=ycenter+j).values,color='black',marker='o',ms=5,alpha=0.5)
    axs[i_].set_title(str(date.date()))


In [ ]:
p = ds['Aerosol_Optical_Depth'].sel(date=slice(date-delta,date+delta)).dropna('wl')

In [ ]:
str(date.date())

In [ ]:
ds['Aerosol_Optical_Depth'].sel(date=slice(date-delta,date+delta)).dropna('wl').plot(x='wl',marker='o',hue='date',lw=0.7)

In [ ]:
from holoviews import streams
import holoviews as hv
import panel as pn
import param
import numpy as np
import xarray as xr
hv.extension('bokeh')
from holoviews import opts

opts.defaults(
    opts.GridSpace(shared_xaxis=True, shared_yaxis=True),
    opts.Image(cmap='binary_r', width=800, height=700),
    opts.Labels(text_color='white', text_font_size='8pt', text_align='left', text_baseline='bottom'),
    opts.Path(color='white'),
    opts.Spread(width=900),
    opts.Overlay(show_legend=True))
# set the parameter for spectra extraction
hv.extension('bokeh')
pn.extension()

raster = img.Rrs#.reset_coords()#.isel(time=-1,drop=True)
ds = hv.Dataset(raster.persist())
im= ds.to(hv.Image, ['x', 'y'], dynamic=True).opts(cmap= 'RdBu_r',colorbar=True)#.hist(bin_range=(0,0.02) ) 
widget = pn.widgets.RangeSlider(start=0, end=1,step=0.001)

jscode = """
    color_mapper.low = cb_obj.value[0];
    color_mapper.high = cb_obj.value[1];
"""
link = widget.jslink(im, code={'value': jscode})
pn.Column(widget, im)